In [ ]:
# Import necessary libraries
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import (
    load_split, add_rul, load_test_rul, last_cycle_per_unit,
    SENSORS, SETTINGS, SENSOR_SYMBOL, SENSOR_NAMES,
)

In [ ]:
# Load data
train = add_rul(load_split("train"))
test = load_split("test")
test_rul = load_test_rul()

print("train:", train.shape, "| engines:", train["unit"].nunique())
print("test: ", test.shape,  "| engines:", test["unit"].nunique())
print("test RUL labels:", len(test_rul))

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
test_rul.head()

In [ ]:
# Engine lifetimes
lifetime = train.groupby("unit")["cycle"].max()
print(lifetime.describe())

* Engines fail at different cycle counts (128 to 362).

In [ ]:
# Check for missing values
print(f'Training set has {int(train.isna().sum().sum())} missing data in total\n'
    f'Test set has {int(test.isna().sum().sum())} missing data in total')

`train` and `test` has no missing value in any column, hence no need for imputation.

In [ ]:
# Data integrity check.
# Check that no cycle skips
g = train.groupby("unit")["cycle"]
assert (g.max() == g.count()).all()

In [ ]:
# Check for test set
g1 = test.groupby("unit")["cycle"]
assert (g1.max() == g1.count()).all()

In [ ]:
# Check that the RUL labels line up with the test engines.
assert test_rul.index.equals(last_cycle_per_unit(test).index)

### Constant sensors
A constant sensor means the sensor reports the exact same value in every row, for every engine, at every point in life. A sensor can only help predict failure if its readings change as the engine degrades. A constant has no variation to learn from, so it carries zero information about RUL. Standardization also computes (x - mean) / std, which divides by zero and produces NaNs.

In [ ]:
n_unique = train[SENSORS].nunique().sort_values()
print(n_unique.head(8))

**Constant Sensors: s1, s5, s10, s16, s18, s19**. They all have a single distinct value across all 20,631 rows, so they carry no information about degradation and will be dropped.  
**s6** has two distinct values. It isn't constant, but it can report only one od two readings. It is also a likely candidate to drop

In [ ]:
train[SETTINGS].nunique().sort_values()

* `op1`:**Altitude**
* `op2`: **Mach number**. Speed relative to the speed of sound.
* `op3`: **Throttle resolver angle**. How hard the pilot pushes the engine

In [ ]:
print(train[["op1", "op2"]].describe().T[["min", "max", "std", "mean"]])
print(train["s6"].value_counts())

`op1`**[Altitude]** only moves beteen -0.0087 and 0.0087 (std ~0.002) and `op2`**[Mach number] between -0.0006 and 0.0006 (std ~0.0003).  
**s6** is almost alway one value. 21.61 appears in 20,225 of 20,631 rows (~98%), and 21.60 appears in 406 rows (~2%).

### VISUALIZATION
1. Sensor Trends VS Remaining Useful Life (RUL)
2. Lifetime plot

In [ ]:
rng = np.random.default_rng(42)
sample_units = rng.choice(train["unit"].unique(), size=5, replace=False)

fig, axes = plt.subplots(7, 3, figsize=(15, 20), sharex=True)
axes = axes.flatten()

for ax, sensor in zip(axes, SENSORS):
    for unit in sample_units:
        unit_df = train[train["unit"] == unit]
        ax.plot(unit_df["rul"], unit_df[sensor], alpha=0.6, label=f"eng{unit}")
    ax.set_title(f"{sensor} ({SENSOR_SYMBOL[sensor]})", fontsize=9)
    ax.grid(True)

axes[0].invert_xaxis()  # shared x-axis: flips every subplot (healthy on the left, failure on the right)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
fig.supxlabel("Remaining useful life (cycles)")
fig.supylabel("Sensor measurement")
fig.suptitle("FD001 sensor trends vs RUL (5 randomly sampled engines)")
fig.tight_layout(rect=[0, 0, 1, 0.96])

fig.savefig("../images/sensor_trends_vs_rul.png", dpi=150, bbox_inches="tight")
plt.show()

* Most degrading sensors stay roughly flat until somewhere around 100 to 125 cycles before failure, then rise or fall, and the change speeds up near the end.
* **s9** and **s14** show the engines starting together, but one engine shoots up, another drifts slightly down and the others barely move. The direction and size of degradation differ per engine.
* The noise is large compared with the trend, which is why smoothing or rolling-window features will matter.

In [ ]:
train[SENSORS + ['rul']].corr()['rul'].sort_values(key=abs, ascending=False)

Computing it between each sensor and RUL gives a first, quick ranking of which sensors track degradation. It's a screening tool, not a final feature selection. It only captures linear relationships, and it's an average over all engines, so it can miss sensors that matter in a more complex way.  

**Sign**: a positive correlation means the sensor is higher when RUL is high (healthy), and negative means it rises as the engine approaches failure.  
* Negative sign (s2, s3, s4, s8, s11, s13, s15, s17): the sensor rises as the engine approaches failure, as seen from the above plot.
* Positive sign (s7, s12, s20, s21): the sensor falls as the engine approaches failure, as seen from the above plot.

**Size**: the larger the absolute value, the more consistently the sensor tracks life remaining. Some sensors should clearly stand out. The ones near zero don't help.

In [ ]:
# Liftime plot
lifetimes = train.groupby("unit")["cycle"].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: distribution
axes[0].hist(lifetimes, bins=15, edgecolor="white")
axes[0].axvline(lifetimes.mean(), linestyle="--", color="red",
                label=f"mean = {lifetimes.mean():.0f}")
axes[0].axvline(lifetimes.median(), linestyle=":", color="black",
                label=f"median = {lifetimes.median():.0f}")
axes[0].set_xlabel("Engine lifetime (cycles)")
axes[0].set_ylabel("Number of engines")
axes[0].set_title("Distribution of lifetimes")
axes[0].legend()

# Panel 2: each engine
sorted_lt = lifetimes.sort_values()
axes[1].bar(range(len(sorted_lt)), sorted_lt.values)
axes[1].set_xlabel("Engines (sorted by lifetime)")
axes[1].set_ylabel("Lifetime (cycles)")
axes[1].set_title("Lifetime of each engine")

fig.suptitle("FD001 training engines: run-to-failure lifetimes")
fig.tight_layout()

fig.savefig("../images/engine_lifetimes.png", dpi=150, bbox_inches="tight")  # save BEFORE show
plt.show()

### Engine lifetimes (FD001 training set)

- Lifetimes range from about 128 to about 362 cycles, a nearly threefold spread
- Mean lifetime is 206.31 cycles and the median is 199 cycles. The most common
  lifetime is around 200 cycles (the tallest histogram bar).
- The distribution is **right-skewed**: most engines fail between roughly 150 and
  250 cycles, with a long tail of a few long-lived engines above 300. The mean sits
  above the median because the tail pulls it up.
- Takeaway: identical engines fail at very different ages, so cycle count alone
  cannot predict failure. The model has to learn from the sensor readings.